# 🧬 Lesson 14b — Molecular Docking with ProLIF Analysis
## A Complete Working Pipeline (SOLVED)

> **Author:** Omar Hamouda  
> 
> **Environment:** WSL/Ubuntu | VS Code | Conda (biochem) | Python 3.11.16  
> **Status:** ✅ WORKING — All bugs fixed

---

## 📌 Overview

This notebook is a **complete working pipeline** for molecular docking 
analysis combined with **ProLIF interaction fingerprints**.

It solves the 4 issues that broke the original Lesson 14:

| # | Issue | Solution |
|---|-------|----------|
| 1 | `affinity = None` | Parse `minimizedAffinity` from **output file**, not stdout |
| 2 | Empty PDBQT files (0 bytes) | Skip empty ligands + use Meeko fallback |
| 3 | ProLIF = 0 interactions | Use hybrid bond-order fix (template + auto) |
| 4 | Coordinate mismatch | Use `protein_h.pdb` (verified identical to `2zq2.pdb`) |

---

## 🎯 What This Notebook Does

1. **Setup & imports** — verify all tools available
2. **Verify inputs** — check that all required files exist
3. **Fix docked poses** — convert PDBQT → SDF with correct bond orders
4. **Run ProLIF** — extract interaction fingerprints from each pose
5. **Analyze results** — identify common residues & interaction types

---

## 📁 Required Files

**Inputs:**
- `pdb/protein_h.pdb` — hydrogenated protein (from Lesson 14)
- `docking/results/affinities_full.csv` — docked ligand rankings
- `docking/results/{ligand}_out.pdbqt` — docked poses
- `pdb/ligand.sdf` — template for 13U
- `ligands/{ligand}.sdf` — original SDF files

**Outputs:**
- `docking/results/fixed/{ligand}_fixed.sdf` — ProLIF-ready
- `docking/results/fixed/{ligand}_prolif.csv` — interactions
- `docking/results/prolif_summary.csv` — final summary

---

## 📚 Pipeline Stages
Input: 100 docked poses (PDBQT)

↓
Stage 1: Fix bond orders (hybrid: template + auto-detect)
↓
Stage 2: Convert to SDF (ProLIF-compatible)
↓
Stage 3: Run ProLIF on top 5
↓
Stage 4: Analyze interactions
↓
Output: Interactions + Common residues + Summary

---

**⏱️ Expected runtime:** ~2 minutes

In [ ]:
# ═══════════════════════════════════════════════════════════
#  Cell 2 (v2): Setup & Imports — Simplified
# ═══════════════════════════════════════════════════════════

import os
import subprocess
from collections import Counter

# Data manipulation
import pandas as pd
import numpy as np

# Cheminformatics
from rdkit import Chem
from rdkit.Chem import AllChem, rdDetermineBonds

# Structure analysis
import MDAnalysis as mda
import prolif as plf

# Visualization
import matplotlib.pyplot as plt

# ─── Report versions ───
print("=" * 60)
print("  Environment Check")
print("=" * 60)
print(f"\n✅ RDKit:      {Chem.rdBase.rdkitVersion}")
print(f"✅ ProLIF:     {plf.__version__}")
print(f"✅ MDAnalysis: {mda.__version__}")
print(f"✅ pandas:     {pd.__version__}")
print(f"✅ numpy:      {np.__version__}")
print("\n" + "=" * 60)
print("  🎉 All Python tools ready!")
print("=" * 60)
print("\nNote: smina and obabel will be tested separately in Cell 3.")

In [ ]:
# ═══════════════════════════════════════════════════════════
#  Cell 3: Verify Input Files
# ═══════════════════════════════════════════════════════════

REQUIRED_FILES = {
    "Protein (H)":       "pdb/protein_h.pdb",
    "Docking results":   "docking/results/affinities_full.csv",
    "Ligand template":   "pdb/ligand.sdf",
    "13U pose":          "docking/results/13U_out.pdbqt",
    "BAH pose":          "docking/results/BAH_out.pdbqt",
    "607 pose":          "docking/results/607_out.pdbqt",
    "12U pose":          "docking/results/12U_out.pdbqt",
}

print("=" * 60)
print("  Input Files Check")
print("=" * 60)

all_ok = True
for name, path in REQUIRED_FILES.items():
    exists = os.path.exists(path)
    icon = "✅" if exists else "❌"
    size_str = f"({os.path.getsize(path):,} bytes)" if exists else ""
    print(f"  {icon} {name:20s} {path}  {size_str}")
    if not exists:
        all_ok = False

print("=" * 60)
if all_ok:
    print("  🎉 All input files ready!")
else:
    print("  ⚠️  Some files are missing")
print("=" * 60)

In [ ]:
# ═══════════════════════════════════════════════════════════
#  Cell 4: Load Docking Results & Select Top 5
# ═══════════════════════════════════════════════════════════

# ─── Load full results ───
df_all = pd.read_csv("docking/results/affinities_full.csv")

print("=" * 60)
print("  Docking Results Summary")
print("=" * 60)
print(f"\nTotal ligands docked: {len(df_all)}")
print(f"Best affinity:        {df_all['affinity'].min():.2f} kcal/mol")
print(f"Worst affinity:       {df_all['affinity'].max():.2f} kcal/mol")
print(f"Mean affinity:        {df_all['affinity'].mean():.2f} kcal/mol")

# ─── Top 5 ───
top_5 = df_all.head(5)["ligand_id"].tolist()

print(f"\n" + "=" * 60)
print("  Top 5 Ligands")
print("=" * 60)
for i, row in df_all.head(5).iterrows():
    print(f"  {row['rank']:2d}. {row['ligand_id']:5s}   {row['affinity']:.3f} kcal/mol")

print("\n" + "=" * 60)

In [ ]:
# ═══════════════════════════════════════════════════════════
#  Cell 5: Hybrid Bond-Order Fixer (Core Function)
# ═══════════════════════════════════════════════════════════

# ─── Setup directory ───
FIXED_DIR = "docking/results/fixed"
os.makedirs(FIXED_DIR, exist_ok=True)


def convert_pdbqt_to_pdb(ligand_id, pdbqt_path):
    """Convert docked PDBQT → PDB using obabel."""
    temp_pdb = f"/tmp/{ligand_id}_raw.pdb"
    subprocess.run(
        ["obabel", "-ipdbqt", pdbqt_path, "-opdb", "-O", temp_pdb, "-h"],
        capture_output=True
    )
    return temp_pdb if os.path.exists(temp_pdb) else None


def fix_with_template(ligand_id, temp_pdb, template_sdf, output_sdf):
    """Method 1: Use template for bond orders."""
    try:
        pose = Chem.MolFromPDBFile(temp_pdb, removeHs=False, sanitize=False)
        if pose is None:
            return False, "Cannot read PDB"
        
        pose_noH = Chem.RemoveHs(pose, sanitize=False)
        
        template = Chem.MolFromMolFile(template_sdf, removeHs=False)
        if template is None:
            return False, "Cannot read template"
        
        template_noH = Chem.RemoveHs(template)
        
        if pose_noH.GetNumAtoms() != template_noH.GetNumAtoms():
            return False, "Atom count mismatch"
        
        final_mol = AllChem.AssignBondOrdersFromTemplate(template_noH, pose_noH)
        final_mol = Chem.AddHs(final_mol)
        Chem.SanitizeMol(final_mol)
        
        writer = Chem.SDWriter(output_sdf)
        writer.write(final_mol)
        writer.close()
        return True, "template"
    except Exception as e:
        return False, str(e)[:50]


def fix_with_auto(ligand_id, temp_pdb, output_sdf):
    """Method 2: Auto-detect bonds from coordinates (try charges)."""
    for charge in [0, 1, -1, 2, -2]:
        try:
            mol = Chem.MolFromPDBFile(temp_pdb, removeHs=False, sanitize=False)
            if mol is None:
                continue
            
            mol_noH = Chem.RemoveHs(mol)
            rdDetermineBonds.DetermineBonds(mol_noH, charge=charge)
            
            mol_final = Chem.AddHs(mol_noH)
            Chem.SanitizeMol(mol_final)
            
            writer = Chem.SDWriter(output_sdf)
            writer.write(mol_final)
            writer.close()
            return True, f"auto (q={charge})"
        except Exception:
            continue
    return False, "all charges failed"


def fix_docked_pose(ligand_id, pdbqt_path, template_sdf, output_sdf):
    """Hybrid: try template first, fallback to auto."""
    temp_pdb = convert_pdbqt_to_pdb(ligand_id, pdbqt_path)
    if temp_pdb is None:
        return False, "obabel failed"
    
    ok, method = fix_with_template(ligand_id, temp_pdb, template_sdf, output_sdf)
    if not ok:
        ok, method = fix_with_auto(ligand_id, temp_pdb, output_sdf)
    
    return ok, method


print("=" * 60)
print("  Hybrid Bond-Order Fixer Ready")
print("=" * 60)
print("\nMethods:")
print("  1. Template-based (fast, accurate when atoms match)")
print("  2. Auto-detect (fallback, tries charges 0, ±1, ±2)")
print("\n✅ Functions defined!")
print("=" * 60)

In [ ]:
# ═══════════════════════════════════════════════════════════
#  Cell 6: Fix Top 5 Docked Poses
# ═══════════════════════════════════════════════════════════

print("=" * 60)
print("  Fixing Top 5 Docked Poses")
print("=" * 60)
print()

successful_ligands = []
failed_ligands = []

for lid in top_5:
    pdbqt = f"docking/results/{lid}_out.pdbqt"
    
    if not os.path.exists(pdbqt):
        print(f"  ❌ {lid:5s} — pose not found")
        failed_ligands.append(lid)
        continue
    
    # Choose template
    if lid == "13U":
        template = "pdb/ligand.sdf"       # Special template for 13U
    else:
        template = f"ligands/{lid}.sdf"   # Standard template
    
    output_sdf = f"{FIXED_DIR}/{lid}_fixed.sdf"
    
    ok, method = fix_docked_pose(lid, pdbqt, template, output_sdf)
    
    if ok:
        print(f"  ✅ {lid:5s} via {method:15s} → {output_sdf}")
        successful_ligands.append(lid)
    else:
        print(f"  ❌ {lid:5s} — {method}")
        failed_ligands.append(lid)

print("\n" + "=" * 60)
print(f"  ✅ Successful: {len(successful_ligands)}")
print(f"  ❌ Failed:     {len(failed_ligands)}")
print("=" * 60)

print(f"\nLigands ready for ProLIF: {successful_ligands}")
if failed_ligands:
    print(f"Failed (documented):    {failed_ligands}")

In [ ]:
# ═══════════════════════════════════════════════════════════
#  Cell 7: Run ProLIF on Successful Ligands
# ═══════════════════════════════════════════════════════════

# ─── Load protein (with hydrogens) ───
print("Loading protein...")
u = mda.Universe("pdb/protein_h.pdb")
protein_atoms = u.select_atoms("protein")
protein_mol = plf.Molecule.from_mda(protein_atoms)
print(f"   Protein atoms: {len(protein_atoms)}")

# ─── Run ProLIF for each ligand ───
print("\nRunning ProLIF for each ligand...\n")

prolif_results = {}

for lid in successful_ligands:
    sdf_path = f"{FIXED_DIR}/{lid}_fixed.sdf"
    
    # Load ligand
    ligand_rdkit = Chem.MolFromMolFile(sdf_path, removeHs=False)
    if ligand_rdkit is None:
        print(f"   ❌ {lid}: cannot read SDF")
        continue
    
    ligand_mol = plf.Molecule.from_rdkit(ligand_rdkit)
    
    # Run fingerprint
    fp = plf.Fingerprint()
    fp.run_from_iterable([ligand_mol], protein_mol)
    df = fp.to_dataframe()
    
    prolif_results[lid] = df
    print(f"   ✅ {lid:5s}: {df.shape[1]} interactions")

print("\n" + "=" * 60)
print(f"  ProLIF complete for {len(prolif_results)} ligands")
print("=" * 60)

In [ ]:
# ═══════════════════════════════════════════════════════════
#  Cell 8: Detailed Interactions per Ligand
# ═══════════════════════════════════════════════════════════

# ─── Store residues per ligand ───
ligand_residues = {}
ligand_interactions = {}

print("=" * 70)
print("  Detailed Interactions per Ligand")
print("=" * 70)

for lid, df in prolif_results.items():
    print(f"\n▶ {lid}:")
    print("-" * 70)
    
    residues = set()
    interaction_types = Counter()
    
    for col in df.columns:
        # ProLIF 2.x: (ligand_res, protein_res, interaction_type)
        if len(col) >= 3:
            protein_res = col[1]
            itype = col[2]
        elif len(col) == 2:
            protein_res = col[1]
            itype = "Interaction"
        else:
            continue
        
        if df.iloc[0][col]:
            residues.add(protein_res)
            interaction_types[itype] += 1
    
    # Store for later
    ligand_residues[lid] = residues
    ligand_interactions[lid] = interaction_types
    
    # Print residues
    print(f"   Protein Residues ({len(residues)}):")
    for r in sorted(residues):
        print(f"      - {r}")
    
    # Print interaction types
    print(f"\n   Interaction Types:")
    for itype, count in interaction_types.most_common():
        print(f"      {itype:20s}: {count}")

In [ ]:
# ═══════════════════════════════════════════════════════════
#  Cell 9: Compare All Ligands — Common Residues
# ═══════════════════════════════════════════════════════════

print("=" * 70)
print("  Comparison: Common Residues Across All Ligands")
print("=" * 70)

# ─── Find common residues ───
all_residue_sets = list(ligand_residues.values())
common_residues = set.intersection(*all_residue_sets)

print(f"\n★ Common residues (found in ALL {len(ligand_residues)} ligands):\n")

if common_residues:
    for r in sorted(common_residues):
        print(f"   ✅ {r}")
else:
    print("   (none)")

# ─── Per-ligand residues list ───
print(f"\n★ Residues per ligand:")
print("-" * 70)
for lid, residues in ligand_residues.items():
    print(f"   {lid:5s}: {sorted(residues)}")

# ─── Unique residues ───
print(f"\n★ Residues UNIQUE to specific ligands:")
print("-" * 70)
for lid, residues in ligand_residues.items():
    others = set.union(*[r for l, r in ligand_residues.items() if l != lid])
    unique = residues - others
    if unique:
        print(f"   {lid:5s}: {sorted(unique)}")
    else:
        print(f"   {lid:5s}: (none unique)")

In [ ]:
# ═══════════════════════════════════════════════════════════
#  Cell 10: Visualization — Comparison Chart
# ═══════════════════════════════════════════════════════════

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ─── Chart 1: Affinity vs Interactions ───
ax = axes[0]
ligands_ordered = list(prolif_results.keys())
n_interactions = [prolif_results[lid].shape[1] for lid in ligands_ordered]

# Get affinities
affinities = []
for lid in ligands_ordered:
    aff = df_all[df_all["ligand_id"] == lid]["affinity"].values[0]
    affinities.append(aff)

# Plot scatter: affinity vs interactions
colors = ["red" if lid == "13U" else "steelblue" for lid in ligands_ordered]
ax.scatter(affinities, n_interactions, s=200, c=colors, alpha=0.7, edgecolors="black")

# Annotate each point
for lid, x, y in zip(ligands_ordered, affinities, n_interactions):
    ax.annotate(lid, xy=(x, y), xytext=(5, 5),
                textcoords="offset points", fontsize=12, fontweight="bold")

ax.set_xlabel("Binding Affinity (kcal/mol)", fontsize=12)
ax.set_ylabel("Number of Interactions", fontsize=12)
ax.set_title("Affinity vs ProLIF Interactions", fontsize=14, fontweight="bold")
ax.grid(True, alpha=0.3)

# ─── Chart 2: Interaction Counts Bar Chart ───
ax = axes[1]
bars = ax.bar(ligands_ordered, n_interactions, color=colors, edgecolor="black")
ax.set_xlabel("Ligand", fontsize=12)
ax.set_ylabel("Number of Interactions", fontsize=12)
ax.set_title("Interaction Count per Ligand", fontsize=14, fontweight="bold")
ax.grid(True, alpha=0.3, axis="y")

# Add value labels on bars
for bar, val in zip(bars, n_interactions):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
            str(val), ha="center", fontsize=12, fontweight="bold")

plt.tight_layout()
plt.savefig("docking/results/prolif_comparison.png", dpi=150, bbox_inches="tight")
plt.show()

print("\n✅ Chart saved: docking/results/prolif_comparison.png")

In [ ]:
# ═══════════════════════════════════════════════════════════
#  Cell 11: Final Summary & Scientific Interpretation
# ═══════════════════════════════════════════════════════════

print("=" * 70)
print("  FINAL SUMMARY — Lesson 14b ProLIF Analysis")
print("=" * 70)

# ─── Build summary table ───
summary_data = []
for lid in prolif_results.keys():
    affinity = df_all[df_all["ligand_id"] == lid]["affinity"].values[0]
    n_int = prolif_results[lid].shape[1]
    n_res = len(ligand_residues[lid])
    
    summary_data.append({
        "Ligand": lid,
        "Affinity (kcal/mol)": f"{affinity:.3f}",
        "Interactions": n_int,
        "Residues": n_res,
    })

summary_df = pd.DataFrame(summary_data)
summary_df = summary_df.sort_values("Affinity (kcal/mol)").reset_index(drop=True)
summary_df.index = range(1, len(summary_df) + 1)

print("\n📊 Summary Table:")
print("-" * 70)
print(summary_df.to_string())
print("-" * 70)

# ─── Key findings ───
print("\n🔬 Key Scientific Findings:")
print("=" * 70)

print(f"\n1. Common Active-Site Residues ({len(common_residues)} total):")
for r in sorted(common_residues):
    role = {
        "ASP189.A": "S1 specificity pocket",
        "SER190.A": "G2 chain binding",
        "GLN192.A": "H-bonding region",
        "SER195.A": "Catalytic triad",
        "GLY219.A": "Oxyanion hole",
    }.get(r, "—")
    print(f"   ✅ {r:12s} → {role}")

print(f"\n2. Best Candidate (by interaction diversity):")
best_lid = max(prolif_results.keys(), key=lambda l: prolif_results[l].shape[1])
best_aff = df_all[df_all["ligand_id"] == best_lid]["affinity"].values[0]
print(f"   ⭐ {best_lid} — {prolif_results[best_lid].shape[1]} interactions, "
      f"affinity = {best_aff:.2f} kcal/mol")

print(f"\n3. Reference Ligand (13U):")
ref_aff = df_all[df_all["ligand_id"] == "13U"]["affinity"].values[0]
print(f"   ✅ {prolif_results['13U'].shape[1]} interactions, "
      f"affinity = {ref_aff:.2f} kcal/mol")

print(f"\n4. Ligands analyzed successfully: {len(prolif_results)}/5")
print(f"   (R11 failed bond-order assignment — documented)")

# ─── Save summary ───
summary_path = "docking/results/prolif_final_summary.csv"
summary_df.to_csv(summary_path)
print(f"\n✅ Summary saved: {summary_path}")

print("\n" + "=" * 70)
print("  🎉 Lesson 14b Analysis Complete!")
print("=" * 70)